----
# SLH-DSA
----

In [1]:
import os

# Use the locally built liboqs install if available
oqs_install_path = r"C:\Users\Admin\_oqs_0.15.0"
if os.path.isdir(oqs_install_path):
    os.environ["OQS_INSTALL_PATH"] = oqs_install_path
    print("Using OQS_INSTALL_PATH:", os.environ["OQS_INSTALL_PATH"])

import oqs
import time
import numpy as np
import matplotlib.pyplot as plt
import csv
import tracemalloc
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding, rsa
from cryptography.hazmat.primitives.serialization import Encoding, NoEncryption, PrivateFormat, PublicFormat

benchmark_output_path = "slh_dsa_benchmark_results.txt"
csv_output_path = "slh_dsa_timing.csv"
rsa_benchmark_output_path = "rsa_3072_benchmark_results.txt"
rsa_csv_output_path = "rsa_3072_timing.csv"


def write_header(file_path, header_text):
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(header_text + "\n")
        f.write("=" * len(header_text) + "\n")

write_header(benchmark_output_path, "SLH-DSA Benchmark Results")
write_header(rsa_benchmark_output_path, "RSA-3072 Benchmark Results")

csv_enabled = True
try:
    with open(csv_output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow(["metric", "run", "time_ms", "peak_kb"])
    with open(rsa_csv_output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow(["metric", "run", "time_ms", "peak_kb"])
except PermissionError:
    print(f"Warning: Cannot write to one or more CSV files (file may be open in another program). CSV logging disabled.")
    csv_enabled = False




def append_to_output(file_path, *lines):
    with open(file_path, "a", encoding="utf-8") as f:
        for line in lines:
            f.write(str(line) + "\n")


def append_to_csv(csv_path, metric, run, time_ms, peak_kb):
    if csv_enabled:
        with open(csv_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f, delimiter=';')
            writer.writerow([metric, run, time_ms, peak_kb])



Using OQS_INSTALL_PATH: C:\Users\Admin\_oqs_0.15.0


c:\Users\Admin\Desktop\benchmark project\Benchmarking_Post-Quantum_Cryptographic_Algorithms\.venv\Lib\site-packages\oqs\__init__.py:1: UserWarning: liboqs version (major, minor) 0.15.0 differs from liboqs-python version 0.14.1
  from oqs.oqs import (


In [2]:
# Test SLH-DSA availability
print("Testing SLH-DSA availability...")
print("\nAvailable SLH-DSA algorithms:")
slh_algs = [alg for alg in oqs.get_enabled_sig_mechanisms() if "SLH" in alg.upper()]
for alg in slh_algs:
    print(f"  ✓ {alg}")

if not slh_algs:
    print("No SLH-DSA algorithms enabled in this OQS build.")
else:
    print("\n✓ SLH-DSA setup successful!")

Testing SLH-DSA availability...

Available SLH-DSA algorithms:
  ✓ SLH_DSA_PURE_SHA2_128S
  ✓ SLH_DSA_PURE_SHA2_128F
  ✓ SLH_DSA_PURE_SHA2_192S
  ✓ SLH_DSA_PURE_SHA2_192F
  ✓ SLH_DSA_PURE_SHA2_256S
  ✓ SLH_DSA_PURE_SHA2_256F
  ✓ SLH_DSA_PURE_SHAKE_128S
  ✓ SLH_DSA_PURE_SHAKE_128F
  ✓ SLH_DSA_PURE_SHAKE_192S
  ✓ SLH_DSA_PURE_SHAKE_192F
  ✓ SLH_DSA_PURE_SHAKE_256S
  ✓ SLH_DSA_PURE_SHAKE_256F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_128S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_128F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_192S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_192F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_256S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_256F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_128S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_128F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_192S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_192F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_256S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_256F
  ✓ SLH_DSA_SHA2_256_PREHASH_SHA2_128S
  ✓ SLH_DSA_SHA2_256_PREHASH_SHA2_128F
  ✓ SLH_DSA_SHA2_256_PREHASH_SHA2_192S
  ✓ SLH_DSA_SHA2

In [ ]:
# Benchmark SLH-DSA keypair generation for a representative parameter set
alg = "SLH_DSA_PURE_SHA2_128F"
if alg not in slh_algs:
    raise ValueError(f"Algorithm {alg} is not enabled. Choose one of: {slh_algs}")

print(f"Benchmarking {alg} keypair generation...")
print("Warming up...")
with oqs.Signature(alg) as sig:
    for _ in range(100):
        _ = sig.generate_keypair()
runs = 20
times_ms = []
peaks_kb = []
with oqs.Signature(alg) as sig:
    for run_index in range(runs):
        start = time.perf_counter()
        tracemalloc.start()
        _ = sig.generate_keypair()
        current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        end = time.perf_counter()
        elapsed_ms = (end - start) * 1000
        peak_kb = peak / 1024
        times_ms.append(elapsed_ms)
        peaks_kb.append(peak_kb)
        append_to_csv(csv_output_path, "keypair_generation", run_index + 1, elapsed_ms, peak_kb)

print(f"Algorithm: {alg}")
print(f"Runs: {runs}")
print(f"Mean keypair time: {np.mean(times_ms):.3f} ms")
print(f"Min keypair time: {np.min(times_ms):.3f} ms")
print(f"Max keypair time: {np.max(times_ms):.3f} ms")
print(f"Mean peak memory: {np.mean(peaks_kb):.3f} KB")
print(f"Max peak memory: {np.max(peaks_kb):.3f} KB")
print("Times (ms):", [round(t, 3) for t in times_ms])
print("Peaks (KB):", [round(p, 3) for p in peaks_kb])

append_to_output(
    benchmark_output_path,
    "\nBenchmarking {alg} keypair generation...".format(alg=alg),
    f"Algorithm: {alg}",
    f"Runs: {runs}",
    f"Mean keypair time: {np.mean(times_ms):.3f} ms",
    f"Min keypair time: {np.min(times_ms):.3f} ms",
    f"Max keypair time: {np.max(times_ms):.3f} ms",
    f"Mean peak memory: {np.mean(peaks_kb):.3f} KB",
    f"Max peak memory: {np.max(peaks_kb):.3f} KB")

Benchmarking SLH_DSA_PURE_SHA2_128F keypair generation...
Warming up...
Algorithm: SLH_DSA_PURE_SHA2_128F
Runs: 10000
Mean keypair time: 7.577 ms
Min keypair time: 2.731 ms
Max keypair time: 230.389 ms
Mean peak memory: 0.540 KB
Max peak memory: 1.734 KB
Times (ms): [2.81, 2.927, 5.725, 4.896, 3.531, 5.385, 2.845, 2.77, 2.885, 5.723, 3.043, 2.78, 2.794, 2.77, 2.98, 2.76, 2.743, 2.832, 2.754, 3.508, 2.782, 4.706, 3.142, 2.772, 2.948, 2.786, 3.44, 2.791, 2.755, 2.934, 2.848, 4.663, 2.812, 2.872, 2.811, 12.409, 2.935, 3.597, 3.101, 21.857, 3.377, 3.669, 3.298, 2.788, 3.095, 2.771, 2.803, 2.973, 2.774, 2.856, 2.768, 2.793, 2.986, 2.751, 4.501, 2.816, 2.994, 2.854, 2.797, 12.735, 2.805, 3.884, 3.288, 3.062, 2.774, 2.829, 3.15, 4.194, 6.515, 5.235, 3.956, 5.878, 4.601, 7.358, 4.901, 5.243, 5.311, 7.047, 7.054, 6.511, 7.285, 7.072, 7.638, 7.792, 7.57, 8.432, 7.934, 8.918, 6.928, 7.14, 6.277, 7.177, 6.805, 7.005, 7.008, 8.224, 8.468, 7.327, 5.385, 7.597, 7.275, 6.209, 8.041, 5.462, 7.131, 7.32

In [ ]:
# Benchmark SLH-DSA signing time
message = b"Hello, world! This is a test message for SLH-DSA signing benchmark."

print(f"Benchmarking {alg} signing...")
print("Warming up...")
with oqs.Signature(alg) as sig:
    pk = sig.generate_keypair()
    context = b""
    for _ in range(100):
        signature = sig.sign_with_ctx_str(message, context)
runs = 20
sign_times_ms = []
sign_peaks_kb = []
with oqs.Signature(alg) as sig:
    pk = sig.generate_keypair()
    context = b""
    for run_index in range(runs):
        start = time.perf_counter()
        tracemalloc.start()
        signature = sig.sign_with_ctx_str(message, context)
        current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        end = time.perf_counter()
        elapsed_ms = (end - start) * 1000
        peak_kb = peak / 1024
        sign_times_ms.append(elapsed_ms)
        sign_peaks_kb.append(peak_kb)
        append_to_csv(csv_output_path, "signing", run_index + 1, elapsed_ms, peak_kb)

print(f"Algorithm: {alg}")
print(f"Runs: {runs}")
print(f"Mean signing time: {np.mean(sign_times_ms):.3f} ms")
print(f"Min signing time: {np.min(sign_times_ms):.3f} ms")
print(f"Max signing time: {np.max(sign_times_ms):.3f} ms")
print(f"Mean peak memory: {np.mean(sign_peaks_kb):.3f} KB")
print(f"Max peak memory: {np.max(sign_peaks_kb):.3f} KB")
print("Signing times (ms):", [round(t, 3) for t in sign_times_ms])
print("Peaks (KB):", [round(p, 3) for p in sign_peaks_kb])

append_to_output(
    benchmark_output_path,
    "\nBenchmarking {alg} signing...".format(alg=alg),
    f"Algorithm: {alg}",
    f"Runs: {runs}",
    f"Mean signing time: {np.mean(sign_times_ms):.3f} ms",
    f"Min signing time: {np.min(sign_times_ms):.3f} ms",
    f"Max signing time: {np.max(sign_times_ms):.3f} ms",
    f"Mean peak memory: {np.mean(sign_peaks_kb):.3f} KB",
    f"Max peak memory: {np.max(sign_peaks_kb):.3f} KB")

Benchmarking SLH_DSA_PURE_SHA2_128F signing...
Warming up...
Algorithm: SLH_DSA_PURE_SHA2_128F
Runs: 10000
Mean signing time: 274.783 ms
Min signing time: 57.035 ms
Max signing time: 826.366 ms
Mean peak memory: 34.147 KB
Max peak memory: 35.066 KB
Signing times (ms): [308.727, 280.51, 322.672, 242.188, 285.1, 280.98, 279.872, 322.631, 308.652, 326.478, 329.116, 294.298, 320.17, 296.008, 252.753, 333.564, 398.33, 308.302, 265.176, 271.277, 264.002, 272.423, 276.279, 235.468, 207.137, 225.58, 322.638, 284.922, 266.391, 230.169, 153.221, 165.065, 279.412, 265.894, 318.484, 268.604, 325.81, 318.354, 417.067, 356.652, 151.43, 227.966, 269.706, 260.213, 204.624, 195.205, 278.571, 250.83, 244.185, 220.8, 170.931, 206.888, 305.99, 223.721, 252.985, 238.449, 219.059, 324.076, 373.832, 233.064, 253.509, 114.849, 161.09, 153.363, 185.074, 119.53, 165.151, 184.931, 183.985, 126.352, 155.404, 197.161, 198.244, 192.087, 150.345, 198.191, 131.815, 196.504, 145.314, 248.051, 250.942, 284.615, 219.691

In [5]:
# SLH-DSA Verification Benchmark
# Note: Verification is currently not working due to a possible bug in the oqs library for SLH-DSA algorithms.
# The regular verify and verify_with_ctx_str methods return False even with correct signatures.
# This may be due to the library version or implementation issues.
# Skipping verification benchmark for now.

In [6]:
# Measure SLH-DSA size metrics
print(f"Measuring {alg} size metrics...")
with oqs.Signature(alg) as sig:
    pk = sig.generate_keypair()
    sk = sig.export_secret_key()
    context = b""
    signature = sig.sign_with_ctx_str(message, context)

print(f"Algorithm: {alg}")
print(f"Public key size: {len(pk)} bytes")
print(f"Secret key size: {len(sk)} bytes")
print(f"Signature size: {len(signature)} bytes")

append_to_output(
    benchmark_output_path,
    "\nMeasuring {alg} size metrics...".format(alg=alg),
    f"Algorithm: {alg}",
    f"Public key size: {len(pk)} bytes",
    f"Secret key size: {len(sk)} bytes",

    f"Signature size: {len(signature)} bytes")

Measuring SLH_DSA_PURE_SHA2_128F size metrics...
Algorithm: SLH_DSA_PURE_SHA2_128F
Public key size: 32 bytes
Secret key size: 64 bytes
Signature size: 17088 bytes


In [ ]:
# RSA-3072 comparison benchmark
rsa_message = b"Hello, world! This is a test message for RSA-3072 benchmark."
rsa_runs = 20

print("Benchmarking RSA-3072 keypair generation...")
print("Warming up...")
for _ in range(100):
    private_key = rsa.generate_private_key(public_exponent=65537, key_size=3072)
rsa_keygen_times = []
rsa_keygen_peaks_kb = []
for run_index in range(rsa_runs):
    start = time.perf_counter()
    tracemalloc.start()
    private_key = rsa.generate_private_key(public_exponent=65537, key_size=3072)
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    end = time.perf_counter()
    elapsed_ms = (end - start) * 1000
    peak_kb = peak / 1024
    rsa_keygen_times.append(elapsed_ms)
    rsa_keygen_peaks_kb.append(peak_kb)
    append_to_csv(rsa_csv_output_path, "keypair_generation", run_index + 1, elapsed_ms, peak_kb)

print(f"Runs: {rsa_runs}")
print(f"Mean keypair time: {np.mean(rsa_keygen_times):.3f} ms")
print(f"Min keypair time: {np.min(rsa_keygen_times):.3f} ms")
print(f"Max keypair time: {np.max(rsa_keygen_times):.3f} ms")
print(f"Mean peak memory: {np.mean(rsa_keygen_peaks_kb):.3f} KB")
print(f"Max peak memory: {np.max(rsa_keygen_peaks_kb):.3f} KB")
print("Times (ms):", [round(t, 3) for t in rsa_keygen_times])
print("Peaks (KB):", [round(p, 3) for p in rsa_keygen_peaks_kb])

append_to_output(
    rsa_benchmark_output_path,
    "\nBenchmarking RSA-3072 keypair generation...",
    f"Runs: {rsa_runs}",
    f"Mean keypair time: {np.mean(rsa_keygen_times):.3f} ms",
    f"Min keypair time: {np.min(rsa_keygen_times):.3f} ms",
    f"Max keypair time: {np.max(rsa_keygen_times):.3f} ms",
    f"Mean peak memory: {np.mean(rsa_keygen_peaks_kb):.3f} KB",
    f"Max peak memory: {np.max(rsa_keygen_peaks_kb):.3f} KB")

print("Benchmarking RSA-3072 signing...")
print("Warming up...")
for _ in range(100):
    signature = private_key.sign(
        rsa_message,
        padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH),
        hashes.SHA256(),
    )
rsa_sign_times = []
rsa_sign_peaks_kb = []
public_key = private_key.public_key()
for run_index in range(rsa_runs):
    start = time.perf_counter()
    tracemalloc.start()
    signature = private_key.sign(
        rsa_message,
        padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH),
        hashes.SHA256(),
    )
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    end = time.perf_counter()
    elapsed_ms = (end - start) * 1000
    peak_kb = peak / 1024
    rsa_sign_times.append(elapsed_ms)
    rsa_sign_peaks_kb.append(peak_kb)
    append_to_csv(rsa_csv_output_path, "signing", run_index + 1, elapsed_ms, peak_kb)

print(f"Runs: {rsa_runs}")
print(f"Mean signing time: {np.mean(rsa_sign_times):.3f} ms")
print(f"Min signing time: {np.min(rsa_sign_times):.3f} ms")
print(f"Max signing time: {np.max(rsa_sign_times):.3f} ms")
print(f"Mean peak memory: {np.mean(rsa_sign_peaks_kb):.3f} KB")
print(f"Max peak memory: {np.max(rsa_sign_peaks_kb):.3f} KB")
print("Signing times (ms):", [round(t, 3) for t in rsa_sign_times])
print("Peaks (KB):", [round(p, 3) for p in rsa_sign_peaks_kb])

append_to_output(
    rsa_benchmark_output_path,
    "\nBenchmarking RSA-3072 signing...",
    f"Runs: {rsa_runs}",
    f"Mean signing time: {np.mean(rsa_sign_times):.3f} ms",
    f"Min signing time: {np.min(rsa_sign_times):.3f} ms",
    f"Max signing time: {np.max(rsa_sign_times):.3f} ms",
    f"Mean peak memory: {np.mean(rsa_sign_peaks_kb):.3f} KB",
    f"Max peak memory: {np.max(rsa_sign_peaks_kb):.3f} KB")

print("Measuring RSA-3072 size metrics...")
rsa_private_bytes = private_key.private_bytes(
    encoding=Encoding.PEM,
    format=PrivateFormat.TraditionalOpenSSL,
    encryption_algorithm=NoEncryption(),
)
rsa_public_bytes = public_key.public_bytes(
    encoding=Encoding.PEM,
    format=PublicFormat.SubjectPublicKeyInfo,
)

print(f"Public key size: {len(rsa_public_bytes)} bytes")
print(f"Private key size: {len(rsa_private_bytes)} bytes")
print(f"Signature size: {len(signature)} bytes")

append_to_output(
    rsa_benchmark_output_path,
    "\nMeasuring RSA-3072 size metrics...",
    f"Public key size: {len(rsa_public_bytes)} bytes",
    f"Private key size: {len(rsa_private_bytes)} bytes",
    f"Signature size: {len(signature)} bytes")

Benchmarking RSA-3072 keypair generation...
Warming up...
Runs: 10000
Mean keypair time: 307.223 ms
Min keypair time: 14.371 ms
Max keypair time: 3270.010 ms
Mean peak memory: 0.046 KB
Max peak memory: 1.219 KB
Times (ms): [550.058, 141.603, 1497.975, 899.786, 819.499, 844.003, 1262.122, 641.192, 606.564, 1928.19, 558.312, 1314.625, 598.956, 186.513, 138.584, 550.59, 177.885, 1109.519, 523.366, 1000.051, 860.954, 340.887, 712.349, 1106.978, 342.631, 449.39, 187.653, 617.122, 509.888, 730.038, 929.535, 743.5, 1233.533, 659.789, 470.299, 924.263, 173.617, 177.504, 429.146, 111.454, 595.398, 1511.76, 1357.415, 1511.768, 1569.869, 52.809, 189.763, 1600.257, 148.87, 626.982, 823.18, 1023.887, 694.871, 447.604, 752.444, 1042.425, 246.586, 139.006, 551.189, 595.035, 263.887, 764.611, 621.954, 325.009, 306.627, 160.399, 527.362, 273.476, 468.385, 890.44, 1185.873, 878.011, 985.921, 269.916, 678.658, 801.596, 1654.282, 723.79, 274.5, 861.63, 734.167, 1724.833, 318.017, 233.781, 1375.768, 448.53